# Notebook 1: Preprocessing — Feature Extraction

**Wildfire Susceptibility Mapping — Muğla Province, Turkey**  
CME434, Karabük University

**Purpose:** Load samplepoints.shp + 15 feature rasters, extract values at each point, save `Inputs.txt` + `Label.txt`  
**Input:** `GIS_Wildfire_Mugla/samplepoints.shp`, `GIS_Wildfire_Mugla/01_elevation.tif` … `15_rainfall.tif`  
**Output:** `Inputs.txt` (1000×15), `Label.txt` (1000,)

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/GIS_Wildfire_Mugla'
print('Drive mounted. Folder:', DRIVE)

## Step 2 — Install / Import Libraries

In [ ]:
!pip install rasterio geopandas --quiet

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
import os

print(f'rasterio {rasterio.__version__}  |  geopandas {gpd.__version__}')

## Step 3 — Load Sample Points

In [ ]:
shp_path = f'{DRIVE}/samplepoints.shp'
gdf = gpd.read_file(shp_path)
print(f'Loaded {len(gdf)} sample points')
print(f'CRS: {gdf.crs}')
print(gdf[['label','geometry']].head())
print('\nClass balance:')
print(gdf['label'].value_counts().rename({1:'Burned', 0:'Unburned'}))

## Step 4 — Define Feature Rasters

In [ ]:
FEATURES = [
    ('elevation',     '01_elevation.tif'),
    ('slope',         '02_slope.tif'),
    ('aspect',        '03_aspect.tif'),
    ('hillshade',     '04_hillshade.tif'),
    ('tpi',           '05_tpi.tif'),
    ('ndvi',          '06_ndvi.tif'),
    ('ndwi',          '07_ndwi.tif'),
    ('evi',           '08_evi.tif'),
    ('nbr',           '09_nbr.tif'),
    ('wind_speed',    '10_wind_speed.tif'),
    ('lst',           '11_lst.tif'),
    ('soil_moisture', '12_soil_moisture.tif'),
    ('dist_roads',    '13_dist_roads.tif'),
    ('dist_urban',    '14_dist_urban.tif'),
    ('rainfall',      '15_rainfall.tif'),
]

for name, fname in FEATURES:
    path = f'{DRIVE}/{fname}'
    exists = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {exists}  {fname}')

## Step 5 — Extract Raster Values at Sample Points

In [ ]:
def extract_values(gdf, raster_path, col_name):
    with rasterio.open(raster_path) as src:
        # Reproject points to raster CRS if needed
        pts = gdf.to_crs(src.crs) if gdf.crs != src.crs else gdf
        coords = [(geom.x, geom.y) for geom in pts.geometry]
        values = [v[0] for v in src.sample(coords)]
    return values

df = gdf[['label']].copy().reset_index(drop=True)

for col_name, fname in FEATURES:
    path = f'{DRIVE}/{fname}'
    vals = extract_values(gdf, path, col_name)
    df[col_name] = vals
    print(f'  Extracted {col_name}: min={min(vals):.3f}  max={max(vals):.3f}')

print(f'\nDataframe shape: {df.shape}')

## Step 6 — Handle Missing Values

In [ ]:
# Replace rasterio nodata sentinels with NaN
df.replace(-9999, np.nan, inplace=True)
df.replace(0, np.nan, inplace=False)  # inspect only — don't drop zeros (valid values)

print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None')

df_clean = df.dropna()
print(f'\nRows before dropna: {len(df)}  |  after: {len(df_clean)}')

## Step 7 — Inspect Dataframe

In [ ]:
print('Shape:', df_clean.shape)
print('\nHead:')
print(df_clean.head())
print('\nDescribe:')
print(df_clean.describe().round(3))
print('\nClass balance:')
counts = df_clean['label'].value_counts()
print(f'  Burned   (1): {counts.get(1, 0)}')
print(f'  Unburned (0): {counts.get(0, 0)}')

## Step 8 — Save Inputs.txt and Label.txt

In [ ]:
feature_cols = [c for c in df_clean.columns if c != 'label']
X = df_clean[feature_cols].values
y = df_clean['label'].values

np.savetxt(f'{DRIVE}/Inputs.txt', X)
np.savetxt(f'{DRIVE}/Label.txt',  y)

print('Saved:')
print(f'  {DRIVE}/Inputs.txt  shape={X.shape}')
print(f'  {DRIVE}/Label.txt   shape={y.shape}')
print('\nFeature column order:')
for i, col in enumerate(feature_cols):
    print(f'  col {i:2d}: {col}')
print('\nDone. Next: 02_ml_training.ipynb')